# Solving CartPole-v1 with Advantage Actor-Critic (A2C)
This notebook demonstrates how to solve the CartPole-v1 environment using the A2C algorithm with PyTorch and gymnasium. We will train an agent, evaluate its performance, and visualize the results.

In [ ]:
# Install required packages (uncomment if running in a fresh environment)
!pip install gymnasium torch numpy matplotlib --quiet

import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random

In [ ]:
# Set up CartPole-v1 environment
env = gym.make('CartPole-v1')
print('Observation space:', env.observation_space)
print('Action space:', env.action_space)
obs_dim = env.observation_space.shape[0]  # number of observation features (state dimensions)
n_actions = env.action_space.n

## Define the A2C Agent
We will implement the actor and critic neural networks, and define the A2C agent logic for action selection, advantage calculation, and loss computation.

In [ ]:
class Actor(nn.Module):
    def __init__(self, obs_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions),
            nn.Softmax(dim=-1)
        )
    def forward(self, x):
        return self.net(x)

class Critic(nn.Module):
    def __init__(self, obs_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.net(x)

class A2CAgent:
    def __init__(self, obs_dim, n_actions, gamma=0.99, lr=1e-3):
        self.actor = Actor(obs_dim, n_actions)
        self.critic = Critic(obs_dim)
        self.gamma = gamma
        self.optimizerA = optim.Adam(self.actor.parameters(), lr=lr)
        self.optimizerC = optim.Adam(self.critic.parameters(), lr=lr)
    def select_action(self, state):
        state = torch.FloatTensor(state)
        probs = self.actor(state)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)
    def compute_returns(self, rewards, dones, next_value):
        R = next_value
        returns = []
        for r, d in zip(reversed(rewards), reversed(dones)):
            R = r + self.gamma * R * (1. - d)
            returns.insert(0, R)
        return returns
    def update(self, states, actions, log_probs, rewards, dones, next_state):
        states = torch.FloatTensor(states)
        actions = torch.LongTensor(actions)
        log_probs = torch.stack(log_probs)
        next_state = torch.FloatTensor(next_state)
        next_value = self.critic(next_state).detach().item()
        returns = self.compute_returns(rewards, dones, next_value)
        returns = torch.FloatTensor(returns)
        values = self.critic(states).squeeze()
        advantage = returns - values
        actor_loss = -(log_probs * advantage.detach()).mean()
        critic_loss = advantage.pow(2).mean()
        self.optimizerA.zero_grad()
        actor_loss.backward()
        self.optimizerA.step()
        self.optimizerC.zero_grad()
        critic_loss.backward()
        self.optimizerC.step()
        return actor_loss.item(), critic_loss.item()

## Train the A2C Agent
We will train the A2C agent on CartPole-v1, logging episode rewards and losses for analysis.

In [ ]:
agent = A2CAgent(obs_dim, n_actions)
num_episodes = 500
max_steps = 500
reward_history = []
actor_losses = []
critic_losses = []

for episode in range(num_episodes):
    state, _ = env.reset()
    states, actions, log_probs, rewards, dones = [], [], [], [], []
    total_reward = 0
    for t in range(max_steps):
        action, log_prob = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        states.append(state)
        actions.append(action)
        log_probs.append(log_prob)
        rewards.append(reward)
        dones.append(done)
        total_reward += reward
        state = next_state
        if done:
            break
    actor_loss, critic_loss = agent.update(states, actions, log_probs, rewards, dones, next_state)
    reward_history.append(total_reward)
    actor_losses.append(actor_loss)
    critic_losses.append(critic_loss)
    if (episode+1) % 50 == 0:
        print(f"Episode {episode+1}, Reward: {total_reward}, Actor Loss: {actor_loss:.3f}, Critic Loss: {critic_loss:.3f}")

plt.plot(reward_history)
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('A2C on CartPole-v1: Episode Rewards')
plt.show()

## Evaluate the Trained Agent
Now we will test the trained agent over several episodes, report the average reward, and visualize its performance.

In [ ]:
test_episodes = 20
test_rewards = []
for ep in range(test_episodes):
    state, _ = env.reset()
    total_reward = 0
    for t in range(max_steps):
        state_tensor = torch.FloatTensor(state)
        with torch.no_grad():
            probs = agent.actor(state_tensor)
        action = torch.argmax(probs).item()
        next_state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        state = next_state
        if terminated or truncated:
            break
    test_rewards.append(total_reward)
    print(f"Test Episode {ep+1}: Reward = {total_reward}")
print(f"\nAverage Test Reward over {test_episodes} episodes: {np.mean(test_rewards):.2f}")